In [1]:
import pandas as pd
from collections import Counter
import re
from fractions import Fraction
import random
import matplotlib.pyplot as plt
import numpy as np
import math
import time
from feval_ttc import load, DatasetType, LLMType
from typing import List, Dict, Any, Optional, Callable, Tuple
from scipy.special import betainc

# Normalizing the answers
def normalize_answer(ans):
    """Normalize an answer string for equality comparison.
    Handles LaTeX \\frac{...}{...}, plain fractions a/b, and numeric strings
    (collapsing 18.00 -> '18', 36/2 -> '18', '8:00' -> '8'). Returns None for
    empty/None input; otherwise returns a canonical string.
    """
    if ans is None:
        return None

    s = str(ans).strip()
    if s == "":
        return None

    # For: "\frac{36}{2}", "\\frac{ 36 }{ 2 }"
    latex_pattern = r'\\+frac\s*\{\s*([+-]?\d+(?:\.\d+)?)\s*\}\s*\{\s*([+-]?\d+(?:\.\d+)?)\s*\}'
    latex_matches = list(re.finditer(latex_pattern, s))
    if latex_matches:
        # Match the last ones
        m = latex_matches[-1]
        num_str = m.group(1)
        den_str = m.group(2)
        try:
            frac = Fraction(num_str) / Fraction(den_str)
        except ZeroDivisionError:
            return None

        if frac.denominator == 1:
            return str(frac.numerator)
        else:
            return f"{frac.numerator}/{frac.denominator}"

    # For: "36/2"、"-3/5"
    m_frac = re.fullmatch(r'([+-]?\d+(?:\.\d+)?)\s*/\s*([+-]?\d+(?:\.\d+)?)', s)
    if m_frac:
        num_str = m_frac.group(1)
        den_str = m_frac.group(2)
        try:
            frac = Fraction(num_str) / Fraction(den_str)
        except ZeroDivisionError:
            return None

        if frac.denominator == 1:
            return str(frac.numerator)
        else:
            return f"{frac.numerator}/{frac.denominator}"

    # For: string + number
    m_num = re.search(r'[-+]?\d+(\.\d+)?', s)
    if not m_num:
        # If no numbers
        return s.lower()

    num_str = m_num.group(0)
    try:
        val = float(num_str)
    except ValueError:
        return s.lower()

    if val.is_integer():
        return str(int(val))      # 18.0 -> '18'
    else:
        # remove additional 0
        return f"{val:.10f}".rstrip('0').rstrip('.')
    
# Load and preprocess the dataset
# Build per-question "answer probability matrix"
def compute_answer_probability_tables_with_meta(
    dataset_name: str,
    model_classes: dict,
    api_path: str = "api_responses.zip",
    N_samples: int = 40,
):
    """
    For a given dataset and a set of LLMs, compute, for each question,
    the probability of each normalized answer for each LLM.

    """

    all_llm_types = [lt for fam in model_classes.values() for lt in fam]

    try:
        ds_type = DatasetType[dataset_name]
    except KeyError:
        ds_type = next(dt for dt in DatasetType if dt.name.lower() == dataset_name.lower())

    dataset, llm_fns = load(ds_type, all_llm_types, api_path=api_path)
    dataset = list(dataset)

    llm_map = {llm_type: fn for llm_type, fn in zip(all_llm_types, llm_fns)}
    results = {}

    for question_id, dataentry in dataset:
        llm_counters = {}
        all_answers = set()

        answers_seq = {}     
        priors_sorted = {}   

        for family, llm_list in model_classes.items():
            for llm_type in llm_list:
                llm_fn = llm_map[llm_type]
                resp = llm_fn(question_id, N=N_samples)

                #  If not None: do normalize
                seq = []
                for cot in resp.cots[:N_samples]:
                    if cot.answer is None:
                        seq.append(None)
                    else:
                        seq.append(normalize_answer(cot.answer))

                llm_name = llm_type.name
                answers_seq[llm_name] = seq

                # Counter（Include None）
                cnt = Counter(seq)
                llm_counters[llm_name] = cnt
                all_answers.update(cnt.keys())

                probs = [c / N_samples for c in cnt.values()]
                priors_sorted[llm_name] = sorted(probs, reverse=True)

        # All unique answers for this question (across all LLMs), includes None
        all_answers = sorted(all_answers, key=lambda x: str(x))
        all_answer_strs = [str(a) for a in all_answers]  

        # Build probability DataFrame: rows = LLM, cols = answer probabilities
        rows_dict = {}
        for llm_name, cnt in llm_counters.items():
            row = {
                ans_str: cnt.get(ans, 0) / N_samples
                for ans, ans_str in zip(all_answers, all_answer_strs)
            }
            rows_dict[llm_name] = row

        df_q = pd.DataFrame.from_dict(rows_dict, orient="index")
        df_q.index.name = "LLM"

        true_answer_norm = normalize_answer(dataentry.answer)
        
        true_mode = {}
        for llm_name, seq in answers_seq.items():
            true_mode[llm_name] = Counter(seq).most_common(1)[0][0]
        
        results[question_id] = {
            "question": dataentry.question,
            "true_answer": dataentry.answer,
            "true_answer_norm": true_answer_norm,
            "prob_table": df_q,
            "answers_seq": answers_seq,
            "priors_sorted": priors_sorted,
            "true_mode": true_mode,
        }

    return results

def bootstrap_answer_probability_tables_with_meta(
    old_results: dict,
    N_samples_new: int,
    seed: int = 0,
):
    """
    Bootstrap (with replacement) from the empirical distribution induced by the
    original 40 samples per (question, LLM). Returns a new results dict with the
    same structure as compute_answer_probability_tables_with_meta.
    """
    rng = np.random.default_rng(seed)

    new_results = {}

    # Keep question iteration order as in old_results (Python 3.7+ preserves dict insertion order)
    for question_id, q in old_results.items():
        answers_seq_old = q["answers_seq"]   # dict: llm_name -> list (length old N, incl None)
        llm_names = list(answers_seq_old.keys())  # preserve order exactly

        answers_seq_new = {}     # llm_name -> new bootstrapped list length N_samples_new
        llm_counters_new = {}    # llm_name -> Counter(new seq)
        all_answers = set()      # union of answers across llms for this question

        # For each LLM: build empirical distribution from old 40 seq, then bootstrap sample
        for llm_name in llm_names:
            seq_old = answers_seq_old[llm_name]
            cnt_old = Counter(seq_old)

            # Empirical support and probs (order doesn't matter for sampling)
            support = list(cnt_old.keys())
            probs = np.array([cnt_old[a] for a in support], dtype=float)
            probs /= probs.sum()  # sum to 1

            # Bootstrap draw from empirical distribution
            sampled = rng.choice(support, size=N_samples_new, replace=True, p=probs).tolist()

            answers_seq_new[llm_name] = sampled

            cnt_new = Counter(sampled)
            llm_counters_new[llm_name] = cnt_new
            all_answers.update(cnt_new.keys())

        # Build prob_table with consistent columns across LLMs
        all_answers = sorted(all_answers, key=lambda x: str(x))
        all_answer_strs = [str(a) for a in all_answers]

        rows_dict = {}
        for llm_name in llm_names:
            cnt_new = llm_counters_new[llm_name]
            row = {
                ans_str: cnt_new.get(ans, 0) / N_samples_new
                for ans, ans_str in zip(all_answers, all_answer_strs)
            }
            rows_dict[llm_name] = row

        df_q = pd.DataFrame.from_dict(rows_dict, orient="index")
        df_q.index.name = "LLM"

        # priors_sorted for the new bootstrapped distribution (descending)
        priors_sorted_new = {}
        for llm_name in llm_names:
            cnt_new = llm_counters_new[llm_name]
            probs_new = [c / N_samples_new for c in cnt_new.values()]
            priors_sorted_new[llm_name] = sorted(probs_new, reverse=True)

        # Pack the same fields as old_results (keep question + true answer metadata unchanged)
        new_results[question_id] = {
            "question": q["question"],
            "true_answer": q["true_answer"],
            "true_answer_norm": q["true_answer_norm"],
            "prob_table": df_q,
            "answers_seq": answers_seq_new,
            "priors_sorted": priors_sorted_new,
        }

    return new_results


# First split the questions in the dataset: use the training dataset to learn the prior,
def split_results_train_test(results, train_ratio=0.7, random_state=42):
    """
    Split results (dict[qid]->obj) into train/test by question_id.

    Returns:
        train_results, test_results
    """
    qids = list(results.keys())
    rng = random.Random(random_state)
    rng.shuffle(qids)

    split_idx = int(len(qids) * train_ratio)
    train_qids = qids[:split_idx]
    test_qids  = qids[split_idx:]

    train_results = {qid: results[qid] for qid in train_qids}
    test_results  = {qid: results[qid] for qid in test_qids}

    return train_results, test_results



def _vec_from(x):
    return list(x.values()) if isinstance(x, dict) else list(x)

def _prior_to_prob_list(prior_obj):
    """
    Convert one question's prior into a list of probabilities in descending order.
    """
    if prior_obj is None:
        return []
    if isinstance(prior_obj, dict):
        probs = list(prior_obj.values())
        probs.sort(reverse=True)
        return probs
    if isinstance(prior_obj, (list, tuple)):
        if len(prior_obj) == 0:
            return []
        # list of floats
        if isinstance(prior_obj[0], (int, float)):
            probs = list(prior_obj)
            probs.sort(reverse=True)
            return probs
        # list of (answer, prob)
        if isinstance(prior_obj[0], (list, tuple)) and len(prior_obj[0]) >= 2:
            probs = [x[1] for x in prior_obj]
            probs.sort(reverse=True)
            return probs
    raise TypeError(f"Unsupported prior format: {type(prior_obj)}")


def build_prior_cache_from_train_results(train_results, llm_name, priors_key="priors_sorted"):
    """
    Build prior_cache from train_results.

    Returns:
        prior_cache = {"K": K, "P_list": P_list}
    """
    raw_vecs = []
    K = 0

    for qid, item in train_results.items():
        prior_obj = item[priors_key][llm_name]
        v = _prior_to_prob_list(prior_obj)  # sorted desc
        raw_vecs.append(v)
        K = max(K, len(v))

    P_list = []
    for v in raw_vecs:
        if len(v) < K:
            v = v + [0.0] * (K - len(v))
        s = sum(v)
        if s > 0:
            v = [x / s for x in v]
        P_list.append(v)

    return {"K": K, "P_list": P_list}


In [2]:
def log_add(log_x: float, log_y: float) -> float:
    """
    Computes log(exp(log_x) + exp(log_y))
    """
    if log_x == float('-inf'): return log_y
    if log_y == float('-inf'): return log_x
    
    if log_x > log_y:
        return log_x + math.log1p(math.exp(log_y - log_x))
    else:
        return log_y + math.log1p(math.exp(log_x - log_y))


def log_sum_exp_list(log_vals: List[float]) -> float:
    """
    Computes log(sum(exp(x))) for a list
    """
    if not log_vals: return float('-inf')
    
    # Filter out -inf
    valid = [x for x in log_vals if x != float('-inf')]
    if not valid: return float('-inf')
    
    mx = max(valid)
    # sum(exp(x - max))
    s = sum(math.exp(x - mx) for x in valid)
    return mx + math.log(s)


# ============================================================

def _compute_log_S_psi_algo2_strict(prior_tail: List[float], 
                                    n_L_bar: int, 
                                    v_d: int, 
                                    c_prime_d: int) -> float:
    """
    Strict implementation of Algorithm 2 in log-space
    """
    N = len(prior_tail)
    
    # Determine m_min
    if v_d > 0:
        m_min = min(N, n_L_bar // v_d)
    else:
        # If cutoff is 0, elements can only be 0.
        # This contributes to 'Tie' (m) if they are 0.
        # But if residual > 0, it's impossible to sum to n_L_bar with 0s.
        if n_L_bar > 0: return float('-inf')
        m_min = N 

    # Initialize DP Table A[m][t] (Log Probabilities)
    # A[m][t] = -inf (0.0 prob) everywhere
    A = [[float('-inf')] * (n_L_bar + 1) for _ in range(m_min + 1)]
    # Base case: A[0][0] = 0.0 (log 1.0)
    A[0][0] = 0.0

    # Pre-compute log factorials for g_j calculation
    log_fact = [0.0] * (v_d + 1)
    for i in range(2, v_d + 1):
        log_fact[i] = math.lgamma(i + 1)

    # DP Loop
    for p_j in prior_tail:
        if p_j <= 0:
            log_p_j = float('-inf')
        else:
            log_p_j = math.log(p_j)

        # Initialize B to -inf
        B = [[float('-inf')] * (n_L_bar + 1) for _ in range(m_min + 1)]
        
        # Precompute log_g_j[r]
        log_g_j = [float('-inf')] * (v_d + 1)
        if log_p_j == float('-inf'):
            log_g_j[0] = 0.0 
        else:
            for r in range(v_d + 1):
                log_g_j[r] = (r * log_p_j) - log_fact[r]

        for m in range(m_min + 1):
            for t in range(n_L_bar + 1):
                log_A_curr = A[m][t]
                if log_A_curr == float('-inf'):
                    continue
                
                # Case 1: Non-tie transition (r < v_d)
                r_limit = min(v_d - 1, n_L_bar - t)
                for r in range(r_limit + 1):
                    # B += A * g  => logB = log_add(logB, logA + logg)
                    term = log_A_curr + log_g_j[r]
                    B[m][t + r] = log_add(B[m][t + r], term)
                
                # Case 2: Tie transition (r = v_d)
                if (t + v_d <= n_L_bar) and (m + 1 <= m_min):
                    term = log_A_curr + log_g_j[v_d]
                    B[m + 1][t + v_d] = log_add(B[m + 1][t + v_d], term)
        
        A = B

    # Apply Weights and Sum
    log_S_psi = float('-inf')
    
    for m in range(m_min + 1):
        log_val = A[m][n_L_bar]
        if log_val == float('-inf'):
            continue
            
        # Weight = 1 / binom(c'_d + m, c'_d)
        binom_val = math.comb(c_prime_d + m, c_prime_d)
        log_weight = -math.log(binom_val)
        #log_binom = math.lgamma(c_prime_d + m + 1) - math.lgamma(c_prime_d + 1) - math.lgamma(m + 1)
        #log_weight = -log_binom
        
        
        term = log_val + log_weight
        log_S_psi = log_add(log_S_psi, term)
        
    # Final Scaling: Multiply by n_L_bar!
    if log_S_psi != float('-inf'):
        log_fact_n = math.lgamma(n_L_bar + 1)
        log_S_psi += log_fact_n
    
    return log_S_psi

###############################################################
# Next: For L=2
###############################################################
def compute_all_S_tilde_L2(K: int, p_values: List[float], n: int, n_1: int) -> List[float]:
    psum = sum(p_values)
    if psum <= 0: return [float('-inf')] * K
    p = [x / psum for x in p_values]
    
    n_L_bar = n - n_1
    v_d = n_1
    c_prime_d = 1 
    
    log_S_results = []
    for i in range(K):
        prior_tail = p[:i] + p[i+1:]
        val = _compute_log_S_psi_algo2_strict(prior_tail, n_L_bar, v_d, c_prime_d)
        log_S_results.append(val)
    return log_S_results

def L2_posterior_tilde(answers: List[str], prior: List[float]) -> float:
    n = len(answers)
    if n == 0: return 0.0
    cnt = Counter(answers)
    n_1 = cnt.most_common(1)[0][1] if cnt else 0
        
    K = len(prior)
    log_S_tilde = compute_all_S_tilde_L2(K, prior, n, n_1)
    
    psum = sum(prior)
    p = [x / psum for x in prior]
    
    log_terms = []
    for i in range(K):
        if p[i] > 0 and log_S_tilde[i] != float('-inf'):
            val = n_1 * math.log(p[i]) + log_S_tilde[i]
            log_terms.append(val)
        else:
            log_terms.append(float("-inf"))
            
            
    max_log = max(log_terms)
    if max_log == float("-inf"): return 0.0
    
    denom = sum(math.exp(lt - max_log) for lt in log_terms)
    prob = math.exp(max_log - (max_log + math.log(denom))) 
    return max(0.0, min(1.0, prob))

###############################################################

def L2_posterior_tilde_uncertain(answers: List[str], prior_cache: Dict[str, Any]) -> float:
    """
    Uncertain-prior posterior for L=2.
    Formula: log P(H1) = LogSumExp_m(log Num_m) - LogSumExp_m(log Den_m)
    """
    n = len(answers)
    if n == 0: return 0.0
    cnt = Counter(answers)
    n_1 = cnt.most_common(1)[0][1] if cnt else 0
    
    K = prior_cache["K"]
    P_list = prior_cache["P_list"]
    
    log_num_terms_m = [] 
    log_den_terms_m = []
    
    for p_raw in P_list:
        # Normalize and sort descending
        psum = sum(p_raw)
        if psum <= 0: continue
        p_m = sorted([x/psum for x in p_raw], reverse=True)
        if len(p_m) < K: p_m.extend([0.0] * (K - len(p_m)))
        p_m = p_m[:K]
        
        # 1. Compute vector of log S_i for this prior m
        log_S_tilde = compute_all_S_tilde_L2(K, p_m, n, n_1)
        
        # 2. Calculate Numerator for this prior: P(H1 | m) * P(O | H1, m)
        # H1 assumes the observed Top-1 corresponds to p_m[0] (the largest prob)
        if p_m[0] > 0 and log_S_tilde[0] != float('-inf'):
            term = n_1 * math.log(p_m[0]) + log_S_tilde[0]
            log_num_terms_m.append(term)
        else:
            log_num_terms_m.append(float("-inf"))
            
        # 3. Calculate Denominator for this prior: Sum_i P(Hi | m) * P(O | Hi, m)
        log_terms_i = []
        for i in range(K):
            if p_m[i] > 0 and log_S_tilde[i] != float('-inf'):
                val = n_1 * math.log(p_m[i]) + log_S_tilde[i]
                log_terms_i.append(val)
            else:
                log_terms_i.append(float("-inf"))
        
        # Evidence for prior m
        log_den_terms_m.append(log_sum_exp_list(log_terms_i))
        
    # Aggregate over all priors (Mixture)
    final_log_num = log_sum_exp_list(log_num_terms_m)
    final_log_den = log_sum_exp_list(log_den_terms_m)
    
    if final_log_den == float("-inf"): return 0.0
    prob = math.exp(final_log_num - final_log_den)
    return max(0.0, min(1.0, prob))

###############################################################
def stop_L2_tilde(answers_all: List[str], prior: List[float], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    prior_sorted = sorted(prior, reverse=True)
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L2_posterior_tilde(answers_n, prior_sorted)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}

###############################################################

def stop_L2_tilde_uncertain(answers_all: List[str], prior_cache: Dict[str, Any], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L2_posterior_tilde_uncertain(answers_n, prior_cache)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}



###############################################################
# Next: For L=3
###############################################################

def compute_all_S_tilde_ij_L3(K: int, prior: List[float], n: int, n_1: int, n_2: int) -> Dict[Tuple[int, int], float]:
    psum = sum(prior)
    if psum <= 0: return {}
    p = [x / psum for x in prior]
    
    n_L_bar = n - n_1 - n_2
    v_d = n_2 
    c_prime_d = 2 if n_1 == n_2 else 1
        
    log_S_ij_results = {}
    for i in range(K):
        for j in range(K):
            if i == j: continue
            prior_tail = [p[k] for k in range(K) if k != i and k != j]
            val = _compute_log_S_psi_algo2_strict(prior_tail, n_L_bar, v_d, c_prime_d)
            log_S_ij_results[(i, j)] = val
    return log_S_ij_results

def L3_posterior_tilde(answers: List[str], prior: List[float]) -> float:
    n = len(answers)
    if n == 0: return 0.0

    cnt = Counter(answers)
    unique_count = len(cnt)
    
    # IF less than 2 unique answer: back to L=2
    if unique_count < 2:
        return L2_posterior_tilde(answers, prior)
    # ======================

    counts = [c for ans, c in cnt.most_common()]
    # Fallback guarantees we have at least 2 counts
    n_1, n_2 = counts[0], counts[1]
    
    K = len(prior)
    psum = sum(prior)
    p = [x / psum for x in prior]
    
    log_S_ij = compute_all_S_tilde_ij_L3(K, prior, n, n_1, n_2)
    log_likelihoods_i = []
    
    for i in range(K):
        if p[i] <= 0:
            log_likelihoods_i.append(float("-inf"))
            continue
            
        terms_j = [] 
        for j in range(K):
            if i == j or p[j] <= 0: continue
            l_s = log_S_ij.get((i, j), float('-inf'))
            if l_s != float('-inf'):
                term = n_1 * math.log(p[i]) + n_2 * math.log(p[j]) + l_s
                terms_j.append(term)
        
        if not terms_j:
            log_likelihoods_i.append(float("-inf"))
        else:
            mx = max(terms_j)
            sum_exp = sum(math.exp(t - mx) for t in terms_j)
            log_likelihoods_i.append(mx + math.log(sum_exp))

    max_total = max(log_likelihoods_i)
    if max_total == float("-inf"): return 0.0
    
    denom = sum(math.exp(l - max_total) for l in log_likelihoods_i)
    prob = math.exp(max_total - (max_total + math.log(denom)))
    return max(0.0, min(1.0, prob))



def L3_posterior_tilde_uncertain(answers: List[str], prior_cache: Dict[str, Any]) -> float:
    n = len(answers)
    if n == 0: return 0.0
    
    # Fallback to L=2 if not enough unique answers
    cnt = Counter(answers)
    if len(cnt) < 2:
        return L2_posterior_tilde_uncertain(answers, prior_cache)
        
    counts = [c for ans, c in cnt.most_common()]
    n_1, n_2 = counts[0], counts[1]
    
    K = prior_cache["K"]
    P_list = prior_cache["P_list"]
    
    log_num_terms_m = []
    log_den_terms_m = []
    
    for p_raw in P_list:
        psum = sum(p_raw)
        if psum <= 0: continue
        p_m = sorted([x/psum for x in p_raw], reverse=True)
        if len(p_m) < K: p_m.extend([0.0] * (K - len(p_m)))
        p_m = p_m[:K]
        
        # 1. Compute Log S_ij for this prior m
        log_S_ij = compute_all_S_tilde_ij_L3(K, p_m, n, n_1, n_2)
        
        # 2. Calculate per-hypothesis likelihoods for this prior
        # log P(Obs | Mode=i, prior=m) = LogSumExp_{j!=i} P(Obs | i, j, m)
        log_likelihoods_i = []
        for i in range(K):
            if p_m[i] <= 0:
                log_likelihoods_i.append(float("-inf"))
                continue
            
            terms_j = []
            for j in range(K):
                if i == j or p_m[j] <= 0: continue
                l_s = log_S_ij.get((i, j), float('-inf'))
                if l_s != float('-inf'):
                    term = n_1 * math.log(p_m[i]) + n_2 * math.log(p_m[j]) + l_s
                    terms_j.append(term)
            
            # Marginalize out j
            log_likelihoods_i.append(log_sum_exp_list(terms_j))
            
        # 3. Denominator for prior m: Sum_i (likelihood_i)
        log_den_m = log_sum_exp_list(log_likelihoods_i)
        log_den_terms_m.append(log_den_m)
        
        # 4. Numerator for prior m: Likelihood of i=0 (Mode matches Top-1)
        # Because p_m is sorted, i=0 is the best candidate.
        if log_likelihoods_i:
            log_num_terms_m.append(log_likelihoods_i[0])
        else:
            log_num_terms_m.append(float("-inf"))
            
    # Mixture Aggregation
    final_log_num = log_sum_exp_list(log_num_terms_m)
    final_log_den = log_sum_exp_list(log_den_terms_m)
    
    if final_log_den == float("-inf"): return 0.0
    prob = math.exp(final_log_num - final_log_den)
    return max(0.0, min(1.0, prob))


# ============================================================

def stop_L3_tilde(answers_all: List[str], prior: List[float], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    prior_sorted = sorted(prior, reverse=True)
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L3_posterior_tilde(answers_n, prior_sorted)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}


def stop_L3_tilde_uncertain(answers_all: List[str], prior_cache: Dict[str, Any], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L3_posterior_tilde_uncertain(answers_n, prior_cache)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}





In [3]:
def split_results_train_test_limited(results, train_ratio=0.7, random_state=42):
    """
    Split results into train/test by question_id.
    """
    qids = list(results.keys())
    rng = random.Random(random_state)
    rng.shuffle(qids)

    split_idx = int(len(qids) * train_ratio)
    train_qids = qids[:split_idx]
    test_qids  = qids[split_idx:]


    train_qids = train_qids[:250]
    test_qids  = test_qids[:250]

    train_results = {qid: results[qid] for qid in train_qids}
    test_results  = {qid: results[qid] for qid in test_qids}

    return train_results, test_results

In [4]:
def run_stop_per_llm_unknown(train_results, test_results, prob_thre, L_approx, llm_names=None):
    """
    No split and bootstrap inside
    """
    test_size = len(test_results)
    
    if llm_names is None:
        first_q = next(iter(test_results.values()))
        llm_names = list(first_q["answers_seq"].keys())

    prior_cache_by_llm = {}
    for llm in llm_names:
        prior_cache_by_llm[llm] = build_prior_cache_from_train_results(
            train_results, llm, priors_key="priors_sorted"
        )

    records_by_llm = {llm: [] for llm in llm_names}
    
    start_time = time.time()
    for done, (qid, obj) in enumerate(test_results.items()):
        answers_seq = obj["answers_seq"]

        for llm in llm_names:
            answers_all = answers_seq[llm]
            cache = prior_cache_by_llm[llm]

            if L_approx == 2:
                out = stop_L2_tilde_uncertain(answers_all, cache, prob_thre)
            elif L_approx == 3:
                out = stop_L3_tilde_uncertain(answers_all, cache, prob_thre)
            else:
                raise ValueError("L_approx must be 2 or 3")
    
            records_by_llm[llm].append({
                "question_id": qid,
                "n_stop": out["n_stop"],
                "final_answer": out["final_answer"],
                "posterior_at_stop": out["posterior"],
                "stopped": out["stopped"],
                "time": out["time"],
            })
            
        if done % 40 == 0 and done > 0:
            print(f"Finished {done}/{test_size} | Time spent: {(time.time()-start_time)/60:.2f} mins")

    tables_by_llm = {}
    for llm, recs in records_by_llm.items():
        tables_by_llm[llm] = pd.DataFrame(recs).sort_values("question_id").set_index("question_id")

    return tables_by_llm


def print_avg_metrics_rp(
    tables_by_llm_or_all,
    results,
    L_approx=None,
    method=None,
    rp=None,          # number of repeat
):

    # normalize input to a list
    if isinstance(tables_by_llm_or_all, list):
        tables_list = tables_by_llm_or_all
        if rp is None:
            rp = len(tables_list)
    else:
        tables_list = [tables_by_llm_or_all]
        if rp is None:
            rp = 1

    if method is None:
        method = f"L={L_approx}" if L_approx is not None else "Unknown"

    # preserve llm order from the first replicate
    llm_order = list(tables_list[0].keys())

    rows = []
    for rep_idx, tables_by_llm in enumerate(tables_list):
        for llm in llm_order:
            df = tables_by_llm[llm]
            for qid, row in df.iterrows():
            
                if qid not in results:
                    continue
                # -------------------

                final_ans = row.get("final_answer", None)
                t = row.get("time", None)
                n_stop = row.get("n_stop", None)

                true_mode = results[qid]["true_mode"][llm]
                true_ans = results[qid]["true_answer_norm"]

                rows.append({
                    "rep": rep_idx,
                    "llm": llm,
                    "qid": qid,
                    "n_stop": n_stop,
                    "time": t,
                    "match_true_mode": 1.0 if final_ans == true_mode else 0.0,
                    "match_true_answer": 1.0 if final_ans == true_ans else 0.0,
                })

    if not rows:
        print(f"Warning: No matching questions found in results for {method}. Check your inputs.")
        return

    df_all = pd.DataFrame(rows)

    # average over all samples
        
    agg = (df_all.groupby("llm", as_index=False)
                  .agg(
                      mean_n_stop=("n_stop", "mean"),
                      std_n_stop=("n_stop", "std"),
                      count_n_stop=("n_stop", "count"),
                      
                      mean_time=("time", "mean"),
                      
                      mode_accuracy=("match_true_mode", "mean"),
                      std_mode=("match_true_mode", "std"),
                      count_mode=("match_true_mode", "count"),
                      
                      answer_accuracy=("match_true_answer", "mean"),
                      std_answer=("match_true_answer", "std"),
                      count_answer=("match_true_answer", "count"),
                  ))

    agg_map = {r["llm"]: r for _, r in agg.iterrows()}

    print(f"Average metrics per LLM ({method}; rp={rp}):")
    for llm in llm_order:
        r = agg_map.get(llm, None)
        if r is None:
            continue
            
        # 95% CI: 1.96 * std
        def get_ci_95(std, count):
            if pd.isna(std) or count <= 1:
                return 0.0
            return 1.96 * (std / np.sqrt(count))
            
        ci_n_stop = get_ci_95(r['std_n_stop'], r['count_n_stop'])
        ci_mode = get_ci_95(r['std_mode'], r['count_mode'])
        ci_answer = get_ci_95(r['std_answer'], r['count_answer'])

        print(
            f"{llm:20s}  "
            f"mean n_stop = {r['mean_n_stop']:.3f} ± {ci_n_stop:.3f}    "
            f"mode accuracy = {r['mode_accuracy']:.3f} ± {ci_mode:.3f}    "
            f"answer accuracy = {r['answer_accuracy']:.3f} ± {ci_answer:.3f}"
        )
    print()    

In [30]:
# The path for the dataset
api_path_global = "/Users/api_responses_fixed.zip"

rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.17 mins
Finished 80/250 | Time spent: 0.32 mins
Finished 120/250 | Time spent: 0.37 mins
Finished 160/250 | Time spent: 0.75 mins
Finished 200/250 | Time spent: 1.22 mins
Finished 240/250 | Time spent: 1.40 mins
L=2 uncertain finished! Time= 1.4036705374717713 mins
Finished 40/250 | Time spent: 0.08 mins
Finished 80/250 | Time spent: 0.14 mins
Finished 120/250 | Time spent: 0.19 mins
Finished 160/250 | Time spent: 0.37 mins
Finished 200/250 | Time spent: 0.68 mins
Finished 240/250 | Time spent: 0.70 mins
L=3 uncertain finished! Time= 0.7090433835983276 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.45 mins
Finished 80/250 | Time spent: 0.74 mins
Finished 120/250 | Time spent: 0.99 mins
Finished 160/250 | Time spent: 1.38 mins
Finished 200/250 | Time spent: 2.15 mins
Finished 240/250 | Time spent: 2.60 mins
L=2 uncertain finished! Time= 2.6007126212120055 mins
Finished 40/250 | Time spent: 0.22 mins
Finished 80/250 | Time spent: 

In [31]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (L=2; rp=5):
Qwen72B25             mean n_stop = 6.408    mode accuracy = 0.995    answer accuracy = 0.881
GPT4oMINI             mean n_stop = 5.394    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=3; rp=5):
Qwen72B25             mean n_stop = 6.227    mode accuracy = 0.995    answer accuracy = 0.881
GPT4oMINI             mean n_stop = 5.128    mode accuracy = 0.996    answer accuracy = 0.858



In [32]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.09 mins
Finished 120/250 | Time spent: 0.12 mins
Finished 160/250 | Time spent: 0.49 mins
Finished 200/250 | Time spent: 0.80 mins
Finished 240/250 | Time spent: 0.82 mins
L=2 uncertain finished! Time= 0.8218570311864217 mins
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.10 mins
Finished 160/250 | Time spent: 0.24 mins
Finished 200/250 | Time spent: 0.49 mins
Finished 240/250 | Time spent: 0.51 mins
L=3 uncertain finished! Time= 0.5091774980227153 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.43 mins
Finished 80/250 | Time spent: 0.59 mins
Finished 120/250 | Time spent: 0.79 mins
Finished 160/250 | Time spent: 1.03 mins
Finished 200/250 | Time spent: 1.51 mins
Finished 240/250 | Time spent: 1.67 mins
L=2 uncertain finished! Time= 1.6755277991294861 mins
Finished 40/250 | Time spent: 0.20 mins
Finished 80/250 | Time spent: 

In [33]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)


Prob. Threshold =  0.975
Average metrics per LLM (L=2; rp=5):
Qwen72B25             mean n_stop = 4.145    mode accuracy = 0.988    answer accuracy = 0.879
GPT4oMINI             mean n_stop = 3.429    mode accuracy = 0.991    answer accuracy = 0.855

Average metrics per LLM (L=3; rp=5):
Qwen72B25             mean n_stop = 4.132    mode accuracy = 0.988    answer accuracy = 0.879
GPT4oMINI             mean n_stop = 3.262    mode accuracy = 0.991    answer accuracy = 0.855



In [35]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )


    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.03 mins
Finished 200/250 | Time spent: 0.03 mins
Finished 240/250 | Time spent: 0.04 mins
L=2 uncertain finished! Time= 0.04181230068206787 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.03 mins
Finished 200/250 | Time spent: 0.03 mins
Finished 240/250 | Time spent: 0.04 mins
L=3 uncertain finished! Time= 0.04017848173777262 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.03 mins
Finished 200/250 | Time spent: 0.03 mins
Finished 240/250 | Time spent: 0.04 mins
L=2 uncertain finished! Time= 0.03978047768274943 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spen

In [36]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)


Prob. Threshold =  0.95
Average metrics per LLM (L=2; rp=5):
Qwen72B25             mean n_stop = 1.000    mode accuracy = 0.953    answer accuracy = 0.863
GPT4oMINI             mean n_stop = 1.000    mode accuracy = 0.969    answer accuracy = 0.855

Average metrics per LLM (L=3; rp=5):
Qwen72B25             mean n_stop = 1.000    mode accuracy = 0.953    answer accuracy = 0.863
GPT4oMINI             mean n_stop = 1.000    mode accuracy = 0.969    answer accuracy = 0.855



In [51]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    #"Qwen": [
        #LLMType.Qwen1B25,
    #    LLMType.Qwen72B25,
    #],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 8.51 mins
Finished 80/250 | Time spent: 18.71 mins
Finished 120/250 | Time spent: 27.33 mins
Finished 160/250 | Time spent: 31.45 mins
Finished 200/250 | Time spent: 36.46 mins
Finished 240/250 | Time spent: 38.63 mins
L=2 uncertain finished! Time= 38.65528661807378 mins
Finished 40/250 | Time spent: 12.59 mins
Finished 80/250 | Time spent: 34.53 mins
Finished 120/250 | Time spent: 46.67 mins
Finished 160/250 | Time spent: 54.87 mins
Finished 200/250 | Time spent: 69.33 mins
Finished 240/250 | Time spent: 72.29 mins
L=3 uncertain finished! Time= 72.33572082916895 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 12.35 mins
Finished 80/250 | Time spent: 21.95 mins
Finished 120/250 | Time spent: 29.19 mins
Finished 160/250 | Time spent: 35.73 mins
Finished 200/250 | Time spent: 46.38 mins
Finished 240/250 | Time spent: 46.68 mins
L=2 uncertain finished! Time= 49.65230520566305 mins
Finished 40/250 | Time spent: 15.86 mins
Finished 80/250

In [52]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.99
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 10.606    mode accuracy = 0.987    answer accuracy = 0.891

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 9.102    mode accuracy = 0.986    answer accuracy = 0.893



In [37]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="AQUA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    

Num. of repeat: 0
Finished 40/77 | Time spent: 12.51 mins
L=2 uncertain finished! Time= 24.64171356757482 mins
Finished 40/77 | Time spent: 23.17 mins
L=3 uncertain finished! Time= 43.23516213496526 mins
Num. of repeat: 1
Finished 40/77 | Time spent: 17.49 mins
L=2 uncertain finished! Time= 24.808825647830965 mins
Finished 40/77 | Time spent: 19.72 mins
L=3 uncertain finished! Time= 29.651564383506773 mins
Num. of repeat: 2
Finished 40/77 | Time spent: 12.87 mins
L=2 uncertain finished! Time= 24.519771385192872 mins
Finished 40/77 | Time spent: 18.30 mins
L=3 uncertain finished! Time= 37.07759770154953 mins
Num. of repeat: 3
Finished 40/77 | Time spent: 13.21 mins
L=2 uncertain finished! Time= 24.0512521982193 mins
Finished 40/77 | Time spent: 28.37 mins
L=3 uncertain finished! Time= 48.49549925327301 mins
Num. of repeat: 4
Finished 40/77 | Time spent: 11.85 mins
L=2 uncertain finished! Time= 20.547806115945182 mins
Finished 40/77 | Time spent: 19.39 mins
L=3 uncertain finished! Time= 

In [38]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.99
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 13.878    mode accuracy = 0.979    answer accuracy = 0.878
Qwen72B25             mean n_stop = 9.364    mode accuracy = 0.992    answer accuracy = 0.873
GPT4oMINI             mean n_stop = 14.506    mode accuracy = 0.969    answer accuracy = 0.860

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 11.101    mode accuracy = 0.979    answer accuracy = 0.878
Qwen72B25             mean n_stop = 8.197    mode accuracy = 0.992    answer accuracy = 0.873
GPT4oMINI             mean n_stop = 11.177    mode accuracy = 0.969    answer accuracy = 0.860



In [40]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="AQUA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    

Num. of repeat: 0
Finished 40/77 | Time spent: 11.95 mins
L=2 uncertain finished! Time= 21.8399307012558 mins
Finished 40/77 | Time spent: 16.82 mins
L=3 uncertain finished! Time= 28.820505348841348 mins
Num. of repeat: 1
Finished 40/77 | Time spent: 11.92 mins
L=2 uncertain finished! Time= 17.471824101607005 mins
Finished 40/77 | Time spent: 15.87 mins
L=3 uncertain finished! Time= 19.60686446825663 mins
Num. of repeat: 2
Finished 40/77 | Time spent: 11.03 mins
L=2 uncertain finished! Time= 19.070643917719522 mins
Finished 40/77 | Time spent: 14.84 mins
L=3 uncertain finished! Time= 31.881511183579764 mins
Num. of repeat: 3
Finished 40/77 | Time spent: 10.46 mins
L=2 uncertain finished! Time= 18.87791196902593 mins
Finished 40/77 | Time spent: 22.53 mins
L=3 uncertain finished! Time= 40.921817715962725 mins
Num. of repeat: 4
Finished 40/77 | Time spent: 8.81 mins
L=2 uncertain finished! Time= 15.956524431705475 mins
Finished 40/77 | Time spent: 14.04 mins
L=3 uncertain finished! Time=

In [41]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.975
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 12.670    mode accuracy = 0.979    answer accuracy = 0.878
Qwen72B25             mean n_stop = 5.771    mode accuracy = 0.987    answer accuracy = 0.875
GPT4oMINI             mean n_stop = 9.792    mode accuracy = 0.956    answer accuracy = 0.849

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 9.460    mode accuracy = 0.979    answer accuracy = 0.878
Qwen72B25             mean n_stop = 5.055    mode accuracy = 0.984    answer accuracy = 0.875
GPT4oMINI             mean n_stop = 7.501    mode accuracy = 0.956    answer accuracy = 0.849



In [42]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="AQUA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    

Num. of repeat: 0
Finished 40/77 | Time spent: 5.67 mins
L=2 uncertain finished! Time= 11.845165415604908 mins
Finished 40/77 | Time spent: 4.12 mins
L=3 uncertain finished! Time= 8.164479649066925 mins
Num. of repeat: 1
Finished 40/77 | Time spent: 7.57 mins
L=2 uncertain finished! Time= 10.20127462943395 mins
Finished 40/77 | Time spent: 12.89 mins
L=3 uncertain finished! Time= 13.995235133171082 mins
Num. of repeat: 2
Finished 40/77 | Time spent: 7.98 mins
L=2 uncertain finished! Time= 12.506378229459127 mins
Finished 40/77 | Time spent: 11.32 mins
L=3 uncertain finished! Time= 18.828963617483776 mins
Num. of repeat: 3
Finished 40/77 | Time spent: 4.44 mins
L=2 uncertain finished! Time= 11.200059119860331 mins
Finished 40/77 | Time spent: 15.44 mins
L=3 uncertain finished! Time= 31.809343298276264 mins
Num. of repeat: 4
Finished 40/77 | Time spent: 7.12 mins
L=2 uncertain finished! Time= 11.703405718008677 mins
Finished 40/77 | Time spent: 11.84 mins
L=3 uncertain finished! Time= 18

In [43]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.95
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 7.631    mode accuracy = 0.956    answer accuracy = 0.865
Qwen72B25             mean n_stop = 4.436    mode accuracy = 0.987    answer accuracy = 0.875
GPT4oMINI             mean n_stop = 8.488    mode accuracy = 0.956    answer accuracy = 0.849

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 5.275    mode accuracy = 0.951    answer accuracy = 0.862
Qwen72B25             mean n_stop = 3.948    mode accuracy = 0.984    answer accuracy = 0.873
GPT4oMINI             mean n_stop = 6.499    mode accuracy = 0.958    answer accuracy = 0.849



In [44]:
# The previous are new results!

In [45]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
     "DeepSeek": [
        LLMType.DeepSeekV3,
    ],
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

Num. of repeat: 0
Finished 40/250 | Time spent: 1.87 mins
Finished 80/250 | Time spent: 7.34 mins
Finished 120/250 | Time spent: 7.49 mins
Finished 160/250 | Time spent: 26.26 mins
Finished 200/250 | Time spent: 26.44 mins
Finished 240/250 | Time spent: 26.86 mins
L=2 uncertain finished! Time= 26.889247051874797 mins
Finished 40/250 | Time spent: 1.97 mins
Finished 80/250 | Time spent: 3.43 mins
Finished 120/250 | Time spent: 3.86 mins
Finished 160/250 | Time spent: 56.54 mins
Finished 200/250 | Time spent: 57.32 mins
Finished 240/250 | Time spent: 58.26 mins
L=3 uncertain finished! Time= 58.29080851872762 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.21 mins
Finished 80/250 | Time spent: 3.61 mins
Finished 120/250 | Time spent: 3.72 mins
Finished 160/250 | Time spent: 20.10 mins
Finished 200/250 | Time spent: 20.49 mins
Finished 240/250 | Time spent: 20.64 mins
L=2 uncertain finished! Time= 20.668462646007537 mins
Finished 40/250 | Time spent: 0.46 mins
Finished 80/250 | Time

In [46]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.99
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 2.702    mode accuracy = 0.997    answer accuracy = 0.972
Qwen72B25             mean n_stop = 3.365    mode accuracy = 1.000    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 4.408    mode accuracy = 0.990    answer accuracy = 0.966
DeepSeekV3            mean n_stop = 2.689    mode accuracy = 0.998    answer accuracy = 0.968

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 2.641    mode accuracy = 0.997    answer accuracy = 0.972
Qwen72B25             mean n_stop = 2.953    mode accuracy = 1.000    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 3.796    mode accuracy = 0.990    answer accuracy = 0.966
DeepSeekV3            mean n_stop = 2.713    mode accuracy = 0.998    answer accuracy = 0.968



In [47]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
     "DeepSeek": [
        LLMType.DeepSeekV3,
    ],
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

Num. of repeat: 0
Finished 40/250 | Time spent: 0.71 mins
Finished 80/250 | Time spent: 0.85 mins
Finished 120/250 | Time spent: 0.97 mins
Finished 160/250 | Time spent: 17.72 mins
Finished 200/250 | Time spent: 17.86 mins
Finished 240/250 | Time spent: 18.24 mins
L=2 uncertain finished! Time= 18.268612416585288 mins
Finished 40/250 | Time spent: 1.55 mins
Finished 80/250 | Time spent: 1.85 mins
Finished 120/250 | Time spent: 2.14 mins
Finished 160/250 | Time spent: 39.16 mins
Finished 200/250 | Time spent: 39.57 mins
Finished 240/250 | Time spent: 40.35 mins
L=3 uncertain finished! Time= 40.37417589823405 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.13 mins
Finished 80/250 | Time spent: 2.78 mins
Finished 120/250 | Time spent: 2.87 mins
Finished 160/250 | Time spent: 18.95 mins
Finished 200/250 | Time spent: 19.08 mins
Finished 240/250 | Time spent: 19.20 mins
L=2 uncertain finished! Time= 19.219967218240104 mins
Finished 40/250 | Time spent: 0.37 mins
Finished 80/250 | Time

In [48]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.975
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 2.485    mode accuracy = 0.997    answer accuracy = 0.972
Qwen72B25             mean n_stop = 1.000    mode accuracy = 0.982    answer accuracy = 0.963
GPT4oMINI             mean n_stop = 3.982    mode accuracy = 0.991    answer accuracy = 0.966
DeepSeekV3            mean n_stop = 1.000    mode accuracy = 0.990    answer accuracy = 0.964

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 2.417    mode accuracy = 0.997    answer accuracy = 0.972
Qwen72B25             mean n_stop = 1.000    mode accuracy = 0.982    answer accuracy = 0.963
GPT4oMINI             mean n_stop = 3.488    mode accuracy = 0.990    answer accuracy = 0.966
DeepSeekV3            mean n_stop = 1.000    mode accuracy = 0.990    answer accuracy = 0.964



In [49]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
     "DeepSeek": [
        LLMType.DeepSeekV3,
    ],
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

Num. of repeat: 0
Finished 40/250 | Time spent: 0.06 mins
Finished 80/250 | Time spent: 0.11 mins
Finished 120/250 | Time spent: 0.17 mins
Finished 160/250 | Time spent: 0.22 mins
Finished 200/250 | Time spent: 0.28 mins
Finished 240/250 | Time spent: 0.33 mins
L=2 uncertain finished! Time= 0.34742736419041953 mins
Finished 40/250 | Time spent: 0.06 mins
Finished 80/250 | Time spent: 0.11 mins
Finished 120/250 | Time spent: 0.17 mins
Finished 160/250 | Time spent: 0.22 mins
Finished 200/250 | Time spent: 0.28 mins
Finished 240/250 | Time spent: 0.33 mins
L=3 uncertain finished! Time= 0.3457662343978882 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.06 mins
Finished 80/250 | Time spent: 0.11 mins
Finished 120/250 | Time spent: 0.17 mins
Finished 160/250 | Time spent: 0.22 mins
Finished 200/250 | Time spent: 0.28 mins
Finished 240/250 | Time spent: 0.33 mins
L=2 uncertain finished! Time= 0.3458183805147807 mins
Finished 40/250 | Time spent: 0.06 mins
Finished 80/250 | Time spent:

In [50]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)
#print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.95
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 1.000    mode accuracy = 0.977    answer accuracy = 0.962
Qwen72B25             mean n_stop = 1.000    mode accuracy = 0.982    answer accuracy = 0.963
GPT4oMINI             mean n_stop = 1.000    mode accuracy = 0.965    answer accuracy = 0.943
DeepSeekV3            mean n_stop = 1.000    mode accuracy = 0.990    answer accuracy = 0.964

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 1.000    mode accuracy = 0.977    answer accuracy = 0.962
Qwen72B25             mean n_stop = 1.000    mode accuracy = 0.982    answer accuracy = 0.963
GPT4oMINI             mean n_stop = 1.000    mode accuracy = 0.965    answer accuracy = 0.943
DeepSeekV3            mean n_stop = 1.000    mode accuracy = 0.990    answer accuracy = 0.964



In [ ]:
# New tests here:

In [53]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    #"Qwen": [
        #LLMType.Qwen1B25,
    #    LLMType.Qwen72B25,
    #],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 8.41 mins
Finished 80/250 | Time spent: 15.80 mins
Finished 120/250 | Time spent: 23.97 mins
Finished 160/250 | Time spent: 27.99 mins
Finished 200/250 | Time spent: 32.64 mins
Finished 240/250 | Time spent: 32.96 mins
L=2 uncertain finished! Time= 32.97447999715805 mins
Finished 40/250 | Time spent: 4.11 mins
Finished 80/250 | Time spent: 17.27 mins
Finished 120/250 | Time spent: 21.38 mins
Finished 160/250 | Time spent: 29.40 mins
Finished 200/250 | Time spent: 43.24 mins
Finished 240/250 | Time spent: 44.38 mins
L=3 uncertain finished! Time= 44.41705361604691 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 11.58 mins
Finished 80/250 | Time spent: 15.65 mins
Finished 120/250 | Time spent: 18.88 mins
Finished 160/250 | Time spent: 25.40 mins
Finished 200/250 | Time spent: 35.29 mins
Finished 240/250 | Time spent: 35.44 mins
L=2 uncertain finished! Time= 35.559984894593555 mins
Finished 40/250 | Time spent: 11.96 mins
Finished 80/250

In [54]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 8.665    mode accuracy = 0.986    answer accuracy = 0.893

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 7.418    mode accuracy = 0.985    answer accuracy = 0.892



In [ ]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    #"Qwen": [
        #LLMType.Qwen1B25,
    #    LLMType.Qwen72B25,
    #],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

Num. of repeat: 0
Finished 40/250 | Time spent: 3.92 mins
Finished 80/250 | Time spent: 8.04 mins
Finished 120/250 | Time spent: 13.15 mins
Finished 160/250 | Time spent: 17.06 mins
Finished 200/250 | Time spent: 21.58 mins
Finished 240/250 | Time spent: 21.69 mins
L=2 uncertain finished! Time= 21.708266683419545 mins
Finished 40/250 | Time spent: 1.23 mins
Finished 80/250 | Time spent: 2.18 mins
Finished 120/250 | Time spent: 5.71 mins
Finished 160/250 | Time spent: 13.29 mins
Finished 200/250 | Time spent: 26.91 mins
Finished 240/250 | Time spent: 27.36 mins
L=3 uncertain finished! Time= 27.394692269961038 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 7.41 mins
Finished 80/250 | Time spent: 8.97 mins
Finished 120/250 | Time spent: 9.81 mins


In [ ]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

In [ ]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="BIGBENCH_DisambiguationQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

In [ ]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

In [ ]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="BIGBENCH_DisambiguationQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

In [ ]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

In [ ]:
rp = 5
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="BIGBENCH_DisambiguationQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')

In [ ]:
print("Prob. Threshold = ", prob_thre)
#print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

In [ ]:
# Rebuttal Code Start Here:

In [5]:
rp = 20
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 1.63 mins
Finished 120/250 | Time spent: 1.66 mins
Finished 160/250 | Time spent: 3.27 mins
Finished 200/250 | Time spent: 4.88 mins
Finished 240/250 | Time spent: 4.90 mins
L=2 uncertain finished! Time= 4.913803950945536 mins
Finished 40/250 | Time spent: 0.04 mins
Finished 80/250 | Time spent: 0.28 mins
Finished 120/250 | Time spent: 0.31 mins
Finished 160/250 | Time spent: 1.41 mins
Finished 200/250 | Time spent: 2.01 mins
Finished 240/250 | Time spent: 2.04 mins
L=3 uncertain finished! Time= 2.0588295340538023 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 1.65 mins
Finished 120/250 | Time spent: 1.68 mins
Finished 160/250 | Time spent: 3.25 mins
Finished 200/250 | Time spent: 4.47 mins
Finished 240/250 | Time spent: 4.50 mins
L=2 uncertain finished! Time= 4.507453103860219 mins
Finished 40/250 | Time spent: 0.04 mins
Finished 80/250 | Time spent: 1.

Finished 240/250 | Time spent: 5.98 mins
L=2 uncertain finished! Time= 5.981618781884511 mins
Finished 40/250 | Time spent: 0.14 mins
Finished 80/250 | Time spent: 0.36 mins
Finished 120/250 | Time spent: 0.39 mins
Finished 160/250 | Time spent: 1.68 mins
Finished 200/250 | Time spent: 2.33 mins
Finished 240/250 | Time spent: 2.76 mins
L=3 uncertain finished! Time= 2.763305354118347 mins
Num. of repeat: 14
Finished 40/250 | Time spent: 0.06 mins
Finished 80/250 | Time spent: 1.74 mins
Finished 120/250 | Time spent: 1.77 mins
Finished 160/250 | Time spent: 3.26 mins
Finished 200/250 | Time spent: 4.56 mins
Finished 240/250 | Time spent: 4.59 mins
L=2 uncertain finished! Time= 4.599361598491669 mins
Finished 40/250 | Time spent: 0.07 mins
Finished 80/250 | Time spent: 4.93 mins
Finished 120/250 | Time spent: 4.96 mins
Finished 160/250 | Time spent: 5.41 mins
Finished 200/250 | Time spent: 5.53 mins
Finished 240/250 | Time spent: 5.58 mins
L=3 uncertain finished! Time= 5.585274064540863 m

In [7]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 3.364 ± 0.296    mode accuracy = 0.994 ± 0.002    answer accuracy = 0.971 ± 0.005

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.842 ± 0.196    mode accuracy = 0.994 ± 0.002    answer accuracy = 0.971 ± 0.005



In [8]:
rp = 20
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']

    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08393087784449259 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=3 uncertain finished! Time= 0.08416846990585328 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08326811790466308 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spen

Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08331331411997477 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=3 uncertain finished! Time= 0.0838396668434143 mins
Num. of repeat: 14
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08325698773066202 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=3 uncertain finished! Time= 0.083505197366

In [9]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.981 ± 0.004    answer accuracy = 0.962 ± 0.005

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.981 ± 0.004    answer accuracy = 0.962 ± 0.005



In [10]:
rp = 20
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']

    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08400166829427083 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=3 uncertain finished! Time= 0.08384258349736531 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08378503322601319 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spen

Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08500585158665976 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.06 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=3 uncertain finished! Time= 0.08522735436757406 mins
Num. of repeat: 14
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=2 uncertain finished! Time= 0.08420556783676147 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.07 mins
Finished 240/250 | Time spent: 0.08 mins
L=3 uncertain finished! Time= 0.08414683341

In [11]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.95
Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.981 ± 0.004    answer accuracy = 0.962 ± 0.005

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.981 ± 0.004    answer accuracy = 0.962 ± 0.005



In [13]:
rp = 20
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="SVAMP",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    
    
    

Num. of repeat: 0
Finished 40/250 | Time spent: 6.73 mins
Finished 80/250 | Time spent: 12.60 mins
Finished 120/250 | Time spent: 12.67 mins
Finished 160/250 | Time spent: 12.73 mins
Finished 200/250 | Time spent: 19.46 mins
Finished 240/250 | Time spent: 19.53 mins
L=2 uncertain finished! Time= 19.539652585983276 mins
Finished 40/250 | Time spent: 15.16 mins
Finished 80/250 | Time spent: 30.77 mins
Finished 120/250 | Time spent: 30.83 mins
Finished 160/250 | Time spent: 30.90 mins
Finished 200/250 | Time spent: 45.18 mins
Finished 240/250 | Time spent: 45.24 mins
L=3 uncertain finished! Time= 45.25509219964345 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 3.18 mins
Finished 80/250 | Time spent: 11.15 mins
Finished 120/250 | Time spent: 13.86 mins
Finished 160/250 | Time spent: 14.02 mins
Finished 200/250 | Time spent: 14.08 mins
Finished 240/250 | Time spent: 14.15 mins
L=2 uncertain finished! Time= 14.16041362285614 mins
Finished 40/250 | Time spent: 1.47 mins
Finished 80/250 

Finished 160/250 | Time spent: 16.71 mins
Finished 200/250 | Time spent: 16.78 mins
Finished 240/250 | Time spent: 16.84 mins
L=2 uncertain finished! Time= 16.858280579249065 mins
Finished 40/250 | Time spent: 4.79 mins
Finished 80/250 | Time spent: 7.56 mins
Finished 120/250 | Time spent: 8.11 mins
Finished 160/250 | Time spent: 8.48 mins
Finished 200/250 | Time spent: 8.55 mins
Finished 240/250 | Time spent: 8.62 mins
L=3 uncertain finished! Time= 8.635210438569386 mins
Num. of repeat: 14
Finished 40/250 | Time spent: 7.00 mins
Finished 80/250 | Time spent: 11.37 mins
Finished 120/250 | Time spent: 14.78 mins
Finished 160/250 | Time spent: 14.94 mins
Finished 200/250 | Time spent: 19.35 mins
Finished 240/250 | Time spent: 19.42 mins
L=2 uncertain finished! Time= 19.43538003762563 mins
Finished 40/250 | Time spent: 37.93 mins
Finished 80/250 | Time spent: 44.64 mins
Finished 120/250 | Time spent: 45.76 mins
Finished 160/250 | Time spent: 46.28 mins
Finished 200/250 | Time spent: 56.02

In [14]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 4.233 ± 0.379    mode accuracy = 0.994 ± 0.002    answer accuracy = 0.949 ± 0.006

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 3.333 ± 0.245    mode accuracy = 0.994 ± 0.002    answer accuracy = 0.949 ± 0.006



In [15]:
rp = 20
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="SVAMP",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.18379468123118084 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=3 uncertain finished! Time= 0.1828233520189921 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.18297468423843383 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent

Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.18231398264567059 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=3 uncertain finished! Time= 0.1821184992790222 mins
Num. of repeat: 14
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.18394938707351685 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=3 uncertain finished! Time= 0.182249728838

In [16]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.976 ± 0.004    answer accuracy = 0.939 ± 0.007

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.976 ± 0.004    answer accuracy = 0.939 ± 0.007



In [17]:
rp = 20
tables_by_llm_L2_u_all = []
tables_by_llm_L3_u_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #]
}


results = compute_answer_probability_tables_with_meta(
    dataset_name="SVAMP",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)

train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )

    #for qid in test_results:
    #    test_results[qid]['answers_seq'] = new_test_results[qid]['answers_seq']


    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_u_all.append(tables_by_llm_L2_u)
    end_L2 = time.time()
    print('L=2 uncertain finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3_u = run_stop_per_llm_unknown(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_u_all.append(tables_by_llm_L3_u)
    end_L3 = time.time()
    print('L=3 uncertain finished! Time=',(end_L3-end_L2)/60,'mins')
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.18266876538594565 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=3 uncertain finished! Time= 0.18300588528315226 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.184149702390035 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent:

Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.1830784519513448 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=3 uncertain finished! Time= 0.18281094630559286 mins
Num. of repeat: 14
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=2 uncertain finished! Time= 0.18730873266855877 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.09 mins
Finished 160/250 | Time spent: 0.12 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.18 mins
L=3 uncertain finished! Time= 0.187275715668

In [18]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_L2_u_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_u_all, test_results, L_approx=3)

Prob. Threshold =  0.95
Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.976 ± 0.004    answer accuracy = 0.939 ± 0.007

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 1.000 ± 0.000    mode accuracy = 0.976 ± 0.004    answer accuracy = 0.939 ± 0.007

